In [111]:
# =========================
# Cell 1) 导入包 & 基础函数（中文注释）
# =========================
import os
import warnings
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

def to_float(s: pd.Series) -> pd.Series:
    """尽可能把一列转成 float；无法转换的变成 NaN"""
    return pd.to_numeric(s, errors="coerce").astype(float)

def zscore(s: pd.Series) -> pd.Series:
    """全样本 z-score 标准化（不分省、不分年）"""
    s = to_float(s)
    mu = np.nanmean(s.values)
    sd = np.nanstd(s.values, ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return s * np.nan
    return (s - mu) / sd

In [112]:
# =========================
# Cell 2) 读取 CSV + 统一主键列名（Province / Year）
# =========================
amr_path = r"C:\Users\lunch\Downloads\amr_rate.csv"
x_path   = r"C:\Users\lunch\Downloads\climate_social_eco.csv"

amr = pd.read_csv(amr_path, encoding="utf-8-sig")
x   = pd.read_csv(x_path,   encoding="utf-8-sig")

def normalize_key_cols(df: pd.DataFrame) -> pd.DataFrame:
    """把省份/年份列统一命名为 Province, Year"""
    col_map = {}
    for c in df.columns:
        cc = str(c).strip()
        ccl = cc.lower()
        if ccl in ["province", "prov", "省份"]:
            col_map[c] = "Province"
        if ccl in ["year", "yr", "年份", "year "]:
            col_map[c] = "Year"
    return df.rename(columns=col_map)

amr = normalize_key_cols(amr)
x   = normalize_key_cols(x)

assert "Province" in amr.columns and "Year" in amr.columns, "amr_rate.csv 缺 Province/Year"
assert "Province" in x.columns   and "Year" in x.columns,   "climate_social_eco.csv 缺 Province/Year"

# 主键清洗
amr["Province"] = amr["Province"].astype(str).str.strip()
x["Province"]   = x["Province"].astype(str).str.strip()
amr["Year"] = pd.to_numeric(amr["Year"], errors="coerce").astype("Int64")
x["Year"]   = pd.to_numeric(x["Year"],   errors="coerce").astype("Int64")

# 只保留 2014-2023
amr = amr[amr["Year"].between(2014, 2023)].copy()
x   = x[x["Year"].between(2014, 2023)].copy()

print("AMR shape:", amr.shape, "| provinces:", amr["Province"].nunique(), "| years:", amr["Year"].min(), "-", amr["Year"].max())
print("X   shape:", x.shape,   "| provinces:", x["Province"].nunique(),   "| years:", x["Year"].min(),   "-", x["Year"].max())

AMR shape: (313, 15) | provinces: 34 | years: 2014 - 2023
X   shape: (310, 31) | provinces: 31 | years: 2014 - 2023


In [113]:
# =========================
# Cell 3) 明确“菌种AMR列”——只从 AMR 表里取（关键修正）
# =========================
# AMR_COLS：amr_rate.csv 里除 Province/Year 外的全部列
AMR_COLS = [c for c in amr.columns if c not in ["Province", "Year"]]

print("AMR 列数（应为你那13个菌种）:", len(AMR_COLS))
print("AMR 列名:", AMR_COLS)

# 把 AMR 列转数值
for c in AMR_COLS:
    amr[c] = to_float(amr[c])

# 简单缺失率检查
miss_amr = amr[AMR_COLS].isna().mean().sort_values(ascending=False)
display(miss_amr.head(20))

AMR 列数（应为你那13个菌种）: 13
AMR 列名: ['MRCNS', 'VREFS', 'VREFM', 'PRSP', 'ERSP', '3GCRKP', 'MRSA', '3GCREC', 'CREC', 'QREC', 'CRPA', 'CRKP', 'CRAB']


PRSP      0.022364
MRCNS     0.019169
VREFS     0.019169
VREFM     0.019169
ERSP      0.019169
3GCRKP    0.019169
MRSA      0.015974
3GCREC    0.015974
CREC      0.015974
QREC      0.015974
CRPA      0.015974
CRKP      0.015974
CRAB      0.015974
dtype: float64

In [114]:
# =========================
# Cell 4) 选择 9 个 X，并“裁剪X表”只保留这些列（关键修正）
# =========================
X_MAP = {
    "TA": "省平均气温",
    "PA": "省平均降水",
    "R1xday": "R1xday",
    "PM25": "PM2.5",
    "MED": "医疗水平",
    "GDP": "GDP",
    "WATER": "城市用水普及率",
    "WASTE": "生活垃圾无害化处理率",
    "AMC": "抗菌药物使用强度",
}

# 如果存在英文缩写列名，则 rename 成中文标准列名
rename_map = {k: v for k, v in X_MAP.items() if k in x.columns and v not in x.columns}
x = x.rename(columns=rename_map)

X_COLS = list(X_MAP.values())
missing_x = [c for c in X_COLS if c not in x.columns]
if missing_x:
    raise KeyError(f"X表缺少这些列：{missing_x}\n当前 x.columns 示例：{list(x.columns)[:50]}")

# 【关键修正】只保留主键 + 9个X，避免混入其它数值列污染后续
x = x[["Province", "Year"] + X_COLS].copy()

print("最终使用的 X 列:", X_COLS)
display(x.head())

最终使用的 X 列: ['省平均气温', '省平均降水', 'R1xday', 'PM2.5', '医疗水平', 'GDP', '城市用水普及率', '生活垃圾无害化处理率', '抗菌药物使用强度']


,Province,Year,省平均气温,省平均降水,R1xday,PM2.5,医疗水平,GDP,城市用水普及率,生活垃圾无害化处理率,抗菌药物使用强度
0,北京,2014,11.802834,459.925228,39.409802,5.74,99,23577.5,100.00,99.6,NaN
1,天津,2014,13.439358,445.971354,33.522428,13.95,56,10749.2,100.00,96.7,NaN
2,河北,2014,10.775661,439.600300,22.180385,179.77,48,25644.1,99.29,86.6,NaN
3,山西,2014,9.330009,533.038397,26.461773,150.68,57,12307.0,98.54,92.1,NaN
4,内蒙古,2014,4.819860,286.822913,10.227644,102.15,62,12377.3,97.79,96.1,NaN


In [115]:
# =========================
# Cell 5) 去重 + 合并面板（Province-Year）
# =========================
DEDUP_STRATEGY = "drop"  # "drop" 或 "mean"

def dedup_panel(df: pd.DataFrame, key=("Province","Year"), strategy="drop") -> pd.DataFrame:
    """处理重复的 Province-Year"""
    if strategy == "drop":
        return df.drop_duplicates(list(key), keep="first").copy()
    if strategy == "mean":
        num_cols = [c for c in df.columns if c not in key and pd.api.types.is_numeric_dtype(df[c])]
        other_cols = [c for c in df.columns if c not in key and c not in num_cols]
        g = df.groupby(list(key), as_index=False)
        out = g[num_cols].mean()
        for c in other_cols:
            out[c] = g[c].first()[c].values
        return out
    raise ValueError("strategy must be 'drop' or 'mean'")

amr2 = dedup_panel(amr, strategy=DEDUP_STRATEGY)
x2   = dedup_panel(x,   strategy=DEDUP_STRATEGY)

df = amr2.merge(x2, on=["Province","Year"], how="inner")
df["Year"] = df["Year"].astype(int)

print("Merged panel shape:", df.shape)
print("Provinces:", df["Province"].nunique(), "| Years:", df["Year"].min(), "-", df["Year"].max())
display(df.head())

Merged panel shape: (310, 24)
Provinces: 31 | Years: 2014 - 2023


,Province,Year,MRCNS,VREFS,VREFM,PRSP,ERSP,3GCRKP,MRSA,3GCREC,CREC,QREC,CRPA,CRKP,CRAB,省平均气温,省平均降水,R1xday,PM2.5,医疗水平,GDP,城市用水普及率,生活垃圾无害化处理率,抗菌药物使用强度
0,北京,2014,82.2,1.6,11.1,0.4,97.0,39.0,49.0,58.8,3.6,61.3,36.1,13.7,64.0,11.802834,459.925228,39.409802,5.74,99,23577.5,100.00,99.6,NaN
1,天津,2014,79.6,0.3,1.9,9.3,96.2,21.5,24.8,51.4,1.5,51.0,22.1,1.7,22.1,13.439358,445.971354,33.522428,13.95,56,10749.2,100.00,96.7,NaN
2,河北,2014,80.3,0.7,2.4,7.0,97.4,47.5,43.4,66.3,2.2,60.1,30.9,6.7,63.0,10.775661,439.600300,22.180385,179.77,48,25644.1,99.29,86.6,NaN
3,山西,2014,70.5,0.6,1.3,1.4,93.8,31.7,24.2,58.4,1.2,55.5,20.4,2.6,52.7,9.330009,533.038397,26.461773,150.68,57,12307.0,98.54,92.1,NaN
4,内蒙古,2014,76.3,0.7,1.4,9.1,95.8,26.1,28.9,55.6,0.6,65.4,20.2,1.9,51.5,4.819860,286.822913,10.227644,102.15,62,12377.3,97.79,96.1,NaN


In [116]:
amc_col = "抗菌药物使用强度"

# 1) 全局缺失率
print("抗菌药物使用强度缺失率：", df[amc_col].isna().mean())

# 2) 哪些省份缺失比较多（Top10）
miss_by_prov = df.groupby("Province")[amc_col].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
display(miss_by_prov.head(10))

# 3) 专门看 2014 是否缺失（因为你要用2015填）
miss_2014 = df[df["Year"]==2014].groupby("Province")[amc_col].apply(lambda s: s.isna().any())
print("2014 缺失的省份数：", int(miss_2014.sum()))
display(miss_2014[miss_2014].head(20))

抗菌药物使用强度缺失率： 0.22903225806451613


Province
西藏     1.0
内蒙古    0.3
上海     0.2
北京     0.2
吉林     0.2
四川     0.2
云南     0.2
宁夏     0.2
安徽     0.2
山东     0.2
Name: 抗菌药物使用强度, dtype: float64

2014 缺失的省份数： 31


Province
上海     True
云南     True
内蒙古    True
北京     True
吉林     True
四川     True
天津     True
宁夏     True
安徽     True
山东     True
山西     True
广东     True
广西     True
新疆     True
江苏     True
江西     True
河北     True
河南     True
浙江     True
海南     True
Name: 抗菌药物使用强度, dtype: bool

In [117]:
import numpy as np
import pandas as pd

def impute_panel_with_log(
    df: pd.DataFrame,
    cols,
    group_col="Province",
    time_col="Year",
    base_year=2014,
    base_year_fill_from=2015,
):
    """
    逐省按年份填补 + 输出日志
    规则：
    1) base_year(2014) 缺失：优先用 base_year_fill_from(2015)；
       若2015也缺 -> 用该省 2014 之后第一个非缺失年份的值（保证能填上）
    2) 其他年份缺失：两侧都有 -> 平均；只有一侧 -> 最近值填（边界也填）
    """
    out = df.copy()
    out = out.sort_values([group_col, time_col]).reset_index(drop=True)

    logs = []

    def first_non_missing_after(g, col, year0):
        """取该省 year0 之后第一个非缺失值及其年份"""
        gg = g[g[time_col] > year0][[time_col, col]].dropna()
        if gg.empty:
            return (np.nan, np.nan)
        row = gg.sort_values(time_col).iloc[0]
        return (int(row[time_col]), float(row[col]))

    def process_group(g: pd.DataFrame, col: str) -> pd.Series:
        s0 = g[col].copy()
        s = s0.copy()

        # ===== 规则1：2014 用 2015；若2015也缺，用2014之后第一个非缺失 =====
        mask_2014 = (g[time_col] == base_year) & s.isna()
        if mask_2014.any():
            # 取2015值
            v2015 = g.loc[g[time_col] == base_year_fill_from, col]
            v2015 = v2015.dropna()
            if len(v2015) > 0:
                fill_year = base_year_fill_from
                fill_val = float(v2015.iloc[0])
                rule = f"A1: {base_year}->{base_year_fill_from}"
            else:
                # 2015也缺：用2014之后第一个非缺失值
                fill_year, fill_val = first_non_missing_after(g, col, base_year)
                rule = f"A2: {base_year}->first_future"
            
            if np.isfinite(fill_val):
                idxs = np.where(mask_2014.to_numpy())[0]
                for i in idxs:
                    logs.append({
                        group_col: g.iloc[i][group_col],
                        time_col: int(g.iloc[i][time_col]),
                        "Variable": col,
                        "Before": np.nan,
                        "After": float(fill_val),
                        "Rule": rule,
                        "Ref1_year": int(fill_year),
                        "Ref1_value": float(fill_val),
                        "Ref2_year": np.nan,
                        "Ref2_value": np.nan
                    })
                s.loc[mask_2014] = fill_val

        # ===== 规则2：其他年份缺失：两侧平均/边界最近值 =====
        prev_val = s.ffill()
        next_val = s.bfill()

        prev_year = g[time_col].where(s.notna()).ffill()
        next_year = g[time_col].where(s.notna()).bfill()

        mask_other = (g[time_col] != base_year) & s.isna()
        if mask_other.any():
            idxs = np.where(mask_other.to_numpy())[0]
            for i in idxs:
                pv, nv = prev_val.iloc[i], next_val.iloc[i]
                py, ny = prev_year.iloc[i], next_year.iloc[i]

                if (not np.isfinite(pv)) and (not np.isfinite(nv)):
                    continue  # 两侧都没值，无法填

                if np.isfinite(pv) and np.isfinite(nv):
                    fill = (pv + nv) / 2.0
                    rule = "B1: mean(prev,next)"
                    ref1y, ref1v = int(py), float(pv)
                    ref2y, ref2v = int(ny), float(nv)
                elif np.isfinite(pv):
                    fill = pv
                    rule = "B2: carry prev (edge)"
                    ref1y, ref1v = int(py), float(pv)
                    ref2y, ref2v = np.nan, np.nan
                else:
                    fill = nv
                    rule = "B3: carry next (edge)"
                    ref1y, ref1v = int(ny), float(nv)
                    ref2y, ref2v = np.nan, np.nan

                logs.append({
                    group_col: g.iloc[i][group_col],
                    time_col: int(g.iloc[i][time_col]),
                    "Variable": col,
                    "Before": np.nan,
                    "After": float(fill),
                    "Rule": rule,
                    "Ref1_year": ref1y,
                    "Ref1_value": ref1v,
                    "Ref2_year": ref2y,
                    "Ref2_value": ref2v
                })
                s.iloc[i] = fill

        return s

    for col in cols:
        out[col] = out.groupby(group_col, group_keys=False).apply(lambda g: process_group(g, col))

    log_df = pd.DataFrame(logs)
    if not log_df.empty:
        log_df = log_df.sort_values([group_col, "Variable", time_col]).reset_index(drop=True)
    return out, log_df

# 你要填补的 X 列（用你真实的 X_COLS）
x_filled, x_log = impute_panel_with_log(x, cols=X_COLS)

print("X 填补日志条数：", len(x_log))
display(x_log[x_log["Variable"]=="抗菌药物使用强度"].head(50))

# 然后再 merge
df = amr.merge(x_filled[["Province","Year"]+X_COLS], on=["Province","Year"], how="inner")

X 填补日志条数： 63


,Province,Year,Variable,Before,After,Rule,Ref1_year,Ref1_value,Ref2_year,Ref2_value
0,上海,2014,抗菌药物使用强度,NaN,52.846010,A1: 2014->2015,2015,52.846010,NaN,NaN
1,上海,2021,抗菌药物使用强度,NaN,45.100000,"B1: mean(prev,next)",2020,50.600000,2022.0,39.60
2,云南,2014,抗菌药物使用强度,NaN,43.522576,A1: 2014->2015,2015,43.522576,NaN,NaN
3,云南,2021,抗菌药物使用强度,NaN,37.900000,"B1: mean(prev,next)",2020,37.800000,2022.0,38.00
4,内蒙古,2014,抗菌药物使用强度,NaN,44.681258,A1: 2014->2015,2015,44.681258,NaN,NaN
5,内蒙古,2018,抗菌药物使用强度,NaN,33.575000,"B1: mean(prev,next)",2017,33.860000,2019.0,33.29
6,内蒙古,2021,抗菌药物使用强度,NaN,37.000000,"B1: mean(prev,next)",2020,32.800000,2022.0,41.20
7,北京,2014,抗菌药物使用强度,NaN,55.247123,A1: 2014->2015,2015,55.247123,NaN,NaN
8,北京,2021,抗菌药物使用强度,NaN,48.800000,"B1: mean(prev,next)",2020,51.100000,2022.0,46.50
9,吉林,2014,抗菌药物使用强度,NaN,44.909242,A1: 2014->2015,2015,44.909242,NaN,NaN


In [118]:
rows_with_na = df[df.isna().any(axis=1)]
print("存在缺失的行数：", rows_with_na.shape[0])
display(rows_with_na)
df.groupby("Province")[AMR_COLS + X_COLS].apply(lambda x: x.isna().mean())

存在缺失的行数： 11


,Province,Year,MRCNS,VREFS,VREFM,PRSP,ERSP,3GCRKP,MRSA,3GCREC,CREC,QREC,CRPA,CRKP,CRAB,省平均气温,省平均降水,R1xday,PM2.5,医疗水平,GDP,城市用水普及率,生活垃圾无害化处理率,抗菌药物使用强度
25,西藏,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.401971,364.989269,11.110497,1.39,41,954.8,89.07,91.2,NaN
56,西藏,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.418497,358.746606,9.994469,1.71,44,1063.6,88.06,91.2,NaN
87,西藏,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.836971,375.122796,7.556777,9.40,45,1197.4,67.57,91.2,NaN
118,西藏,2017,84.5,NaN,NaN,NaN,NaN,NaN,52.0,57.1,0.3,44.1,17.2,0.6,65.2,-1.963908,402.649259,11.069758,4.92,49,1381.0,92.80,95.4,NaN
121,青海,2017,65.2,0.0,0.0,NaN,90.6,14.1,34.8,47.5,0.9,50.4,10.0,0.3,23.3,-2.187987,373.707449,7.267916,11.44,70,2517.0,98.93,94.8,34.11
149,西藏,2018,NaN,0.0,0.0,3.0,67.4,47.8,45.0,51.8,2.8,43.0,12.6,1.1,76.2,-2.227898,384.218142,7.776219,5.45,55,1583.5,85.90,96.0,NaN
180,西藏,2019,77.3,0.0,0.0,5.6,72.4,41.3,44.5,58.4,0.1,34.4,7.1,0.6,24.4,-2.377976,396.884790,10.501694,13.52,60,1750.3,95.03,98.3,NaN
211,西藏,2020,82.9,0.0,0.0,6.0,87.3,36.7,46.0,55.5,0.2,34.4,10.3,0.2,31.5,-2.398619,370.509886,9.063721,1.02,62,1956.5,98.78,99.7,NaN
242,西藏,2021,81.8,0.0,1.3,3.1,84.0,35.2,48.5,54.2,0.2,38.2,10.6,1.6,42.0,-2.054028,359.942807,7.346857,0.83,70,2145.1,98.90,99.7,NaN
273,西藏,2022,84.6,0.0,0.7,2.5,80.9,37.3,44.0,59.8,0.4,42.6,11.6,3.4,41.3,-2.105214,377.205739,8.114036,0.73,73,2235.4,99.70,99.8,NaN


,MRCNS,VREFS,VREFM,PRSP,ERSP,3GCRKP,MRSA,3GCREC,CREC,QREC,CRPA,CRKP,CRAB,省平均气温,省平均降水,R1xday,PM2.5,医疗水平,GDP,城市用水普及率,生活垃圾无害化处理率,抗菌药物使用强度
Province,,,,,,,,,,,,,,,,,,,,,,
上海,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
云南,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
内蒙古,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
北京,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
吉林,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
四川,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
天津,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
宁夏,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
安徽,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [119]:
# =========================
# Cell 6) 标准化设置 + 构造面板索引
# =========================
STANDARDIZE_X = True   # 建议 True：让不同X的系数可比较
STANDARDIZE_Y = False  # 单菌种耐药率一般不强制z-score；你也可以改 True

if STANDARDIZE_X:
    for c in X_COLS:
        df[c] = zscore(df[c])

# 设置面板索引
panel = df.set_index(["Province","Year"]).sort_index()

print("面板索引:", panel.index.names, "| shape:", panel.shape)

面板索引: ['Province', 'Year'] | shape: (310, 22)


In [120]:
# =========================
# Cell 7) 循环跑“单菌种”双固定效应回归（核心）
#   Y(菌种AMR) ~ 9个X + 省份FE + 年份FE
#   标准误：按省份聚类（cluster_entity=True）
# =========================
def fit_fe_one(panel_df: pd.DataFrame, y_col: str, x_cols: list[str]):
    """对一个菌种 y_col 跑双固定效应回归"""
    Y = to_float(panel_df[y_col])
    X = panel_df[x_cols].copy()

    if STANDARDIZE_Y:
        Y = zscore(Y)

    tmp = pd.concat([Y.rename(y_col), X], axis=1).dropna()

    # 如果样本太少，跳过
    if tmp.shape[0] < 80:
        return None

    mod = PanelOLS(tmp[y_col], tmp[x_cols], entity_effects=True, time_effects=True)
    res = mod.fit(cov_type="clustered", cluster_entity=True)
    return res

def summarize_res(res, amr_name: str, x_cols: list[str]) -> pd.DataFrame:
    """把回归结果整理成表格（coef/CI/p/N/R2within）"""
    rows = []
    for x in x_cols:
        b  = float(res.params.get(x, np.nan))
        se = float(res.std_errors.get(x, np.nan))
        p  = float(res.pvalues.get(x, np.nan))
        lo, hi = b - 1.96*se, b + 1.96*se
        rows.append({
            "AMR": amr_name,
            "Predictor": x,
            "Coef": b,
            "CI_lo": lo,
            "CI_hi": hi,
            "p": p,
            "N": int(res.nobs),
            "R2_within": float(res.rsquared_within),
        })
    return pd.DataFrame(rows)

all_rows = []
skipped = []

for y in AMR_COLS:
    res = fit_fe_one(panel, y, X_COLS)
    if res is None:
        skipped.append(y)
        continue
    all_rows.append(summarize_res(res, y, X_COLS))

res_long = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
print("完成：成功建模的菌种数 =", res_long["AMR"].nunique() if not res_long.empty else 0)
print("跳过（样本太少）:", skipped)
display(res_long.head(20))

完成：成功建模的菌种数 = 13
跳过（样本太少）: []


,AMR,Predictor,Coef,CI_lo,CI_hi,p,N,R2_within
0,MRCNS,省平均气温,-2.941425,-16.542942,10.660092,0.672027,300,-0.573236
1,MRCNS,省平均降水,-0.803923,-2.574566,0.966721,0.374370,300,-0.573236
2,MRCNS,R1xday,-0.228156,-0.723136,0.266825,0.367155,300,-0.573236
3,MRCNS,PM2.5,-0.343702,-1.714825,1.027421,0.623630,300,-0.573236
4,MRCNS,医疗水平,2.144193,-2.223712,6.512098,0.336892,300,-0.573236
5,MRCNS,GDP,-1.355122,-3.946287,1.236044,0.306329,300,-0.573236
6,MRCNS,城市用水普及率,0.888728,-0.553745,2.331201,0.228340,300,-0.573236
7,MRCNS,生活垃圾无害化处理率,-0.193549,-1.114030,0.726933,0.680596,300,-0.573236
8,MRCNS,抗菌药物使用强度,-0.055783,-0.703006,0.591440,0.865989,300,-0.573236
9,VREFS,省平均气温,1.124849,-0.140559,2.390256,0.082678,300,0.278565


In [121]:
# =========================
# Cell 8) 专门查看：降水 + 抗菌药物使用强度 在“各菌种”上的结果
# =========================
focus_vars = ["R1xday", "抗菌药物使用强度"]
focus = res_long[res_long["Predictor"].isin(focus_vars)].copy()

# =========================
# 修复版：先用数值 p 排序，再做文本格式化与展示
# =========================

def p_text(p):
    """把 p 值格式化成论文风格字符串"""
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "<0.0001"
    return f"{p:.4f}".rstrip("0").rstrip(".")

# 1) 先复制一份，避免 SettingWithCopy 警告
focus2 = focus.copy()

# 2) 先按数值 p 排序（这里用的是原始数值列 p）
focus2 = focus2.sort_values(["Predictor", "p"], ascending=[True, True])

# 3) 再生成展示用的文本列
focus2["Coef_txt"] = focus2["Coef"].map(lambda v: f"{v:.4f}".rstrip("0").rstrip("."))
focus2["CI_txt"]   = focus2.apply(lambda r: f"{r['CI_lo']:.4f}–{r['CI_hi']:.4f}", axis=1)
focus2["p_txt"]    = focus2["p"].map(p_text)

# 4) 最后只保留你想展示的列（这时候就不需要 p 了）
focus_out = focus2[["AMR","Predictor","Coef_txt","CI_txt","p_txt","N","R2_within"]].copy()

display(focus_out.head(80))

,AMR,Predictor,Coef_txt,CI_txt,p_txt,N,R2_within
101,CRKP,R1xday,0.4405,-0.0758–0.9567,0.0957,300,0.206041
110,CRAB,R1xday,-0.4705,-1.1308–0.1898,0.1637,300,0.260929
29,PRSP,R1xday,-0.2407,-0.6435–0.1621,0.2426,299,-1.447247
47,3GCRKP,R1xday,0.2703,-0.2967–0.8372,0.3511,300,-0.182813
2,MRCNS,R1xday,-0.2282,-0.7231–0.2668,0.3672,300,-0.573236
92,CRPA,R1xday,0.1275,-0.1716–0.4266,0.4043,300,0.111687
11,VREFS,R1xday,0.0184,-0.0332–0.0699,0.4854,300,0.278565
20,VREFM,R1xday,-0.069,-0.2708–0.1328,0.5032,300,-0.170894
38,ERSP,R1xday,-0.1009,-0.6102–0.4084,0.6981,300,0.153687
56,MRSA,R1xday,0.0584,-0.3344–0.4512,0.771,300,-0.519696


In [122]:
# =========================
# Cell 9) 导出结果
# =========================
OUT_DIR = "outputs_single_species_fe"
os.makedirs(OUT_DIR, exist_ok=True)

csv_path  = os.path.join(OUT_DIR, "single_species_FE_long_FIXED.csv")
xlsx_path = os.path.join(OUT_DIR, "single_species_FE_long_FIXED.xlsx")

res_long.to_csv(csv_path, index=False, encoding="utf-8-sig")
res_long.to_excel(xlsx_path, index=False)

print("已保存：", csv_path)
print("已保存：", xlsx_path)

已保存： outputs_single_species_fe\single_species_FE_long_FIXED.csv
已保存： outputs_single_species_fe\single_species_FE_long_FIXED.xlsx


In [123]:
# =========================
# 你手动选择要进入模型的自变量（想保留哪些就写哪些）
# =========================

MY_X = [
    "PM2.5",
    "GDP",
    "抗菌药物使用强度",
    "R1xday",
    "省平均气温",
    "省平均降水",
    "医疗水平",
    "城市用水普及率",
    "生活垃圾无害化处理率",
]

In [124]:
# =========================
# 用 MY_X 跑每个菌种的双固定效应回归，并汇总到 res_all
# =========================
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

# 确保面板索引正确（只做一次）
panel = df.set_index(["Province", "Year"]).sort_index()

results = []

for amr in AMR_COLS:

    # 1) 构建该菌种的回归数据（只取你选的 MY_X）
    tmp = panel[[amr] + MY_X].dropna()

    # 2) 如果样本太少，跳过（避免结果不稳）
    if tmp.shape[0] < 80:
        continue

    y = tmp[amr]
    X = tmp[MY_X]

    # 3) 双固定效应（省份FE + 年份FE），省份聚类稳健SE
    # mod = PanelOLS(y, X, entity_effects=True, time_effects=True)
    mod = PanelOLS(y, X, entity_effects=False, time_effects=True)
    res = mod.fit(cov_type="clustered", cluster_entity=True)

    # 4) 把每个自变量的结果整理出来
    for var in MY_X:
        b  = float(res.params[var])
        se = float(res.std_errors[var])
        p  = float(res.pvalues[var])

        ci_lo = b - 1.96 * se
        ci_hi = b + 1.96 * se

        results.append({
            "AMR": amr,
            "Predictor": var,
            "Coef": b,
            "CI_lo": ci_lo,
            "CI_hi": ci_hi,
            "p": p,
            "N": int(res.nobs),
            "R2_within": float(res.rsquared_within),
            "Model_X": "+".join(MY_X)  # 记录当前模型用了哪些变量（方便你比较不同尝试）
        })

res_all = pd.DataFrame(results)
print("res_all 行数：", len(res_all))
display(res_all.head(20))

res_all 行数： 117


,AMR,Predictor,Coef,CI_lo,CI_hi,p,N,R2_within,Model_X
0,MRCNS,PM2.5,0.374794,-0.664806,1.414395,0.480392,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
1,MRCNS,GDP,1.265906,0.048390,2.483422,0.042496,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
2,MRCNS,抗菌药物使用强度,0.528074,-0.485117,1.541264,0.307873,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
3,MRCNS,R1xday,-0.286027,-1.442874,0.870821,0.628335,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
4,MRCNS,省平均气温,0.245569,-1.810148,2.301287,0.815050,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
5,MRCNS,省平均降水,0.445317,-1.264158,2.154791,0.610047,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
6,MRCNS,医疗水平,1.799386,0.797145,2.801626,0.000505,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
7,MRCNS,城市用水普及率,-1.078214,-2.713132,0.556703,0.197210,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
8,MRCNS,生活垃圾无害化处理率,0.279278,-0.944207,1.502762,0.654933,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
9,VREFS,PM2.5,0.016426,-0.056770,0.089622,0.660384,300,-0.052223,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...


In [125]:
# Cell 1：准备（检查 res_all 结构）
import numpy as np
import pandas as pd

# res_all 必须包含这些列
need_cols = {"AMR","Predictor","Coef","CI_lo","CI_hi","p"}
missing = need_cols - set(res_all.columns)
if missing:
    raise ValueError(f"res_all 缺少列：{missing}，请先确认你的回归汇总表是否正确生成。")

print("res_all 行数:", len(res_all))
display(res_all.head())

res_all 行数: 117


,AMR,Predictor,Coef,CI_lo,CI_hi,p,N,R2_within,Model_X
0,MRCNS,PM2.5,0.374794,-0.664806,1.414395,0.480392,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
1,MRCNS,GDP,1.265906,0.048390,2.483422,0.042496,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
2,MRCNS,抗菌药物使用强度,0.528074,-0.485117,1.541264,0.307873,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
3,MRCNS,R1xday,-0.286027,-1.442874,0.870821,0.628335,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...
4,MRCNS,省平均气温,0.245569,-1.810148,2.301287,0.815050,300,-0.520951,PM2.5+GDP+抗菌药物使用强度+R1xday+省平均气温+省平均降水+医疗水平+城市用...


In [126]:
# Cell 2：生成“显著性星号”与格式化文本
# =========================
# 13菌种 × 9变量：系数矩阵（带显著性）
# =========================

def sig_star(p):
    """显著性星号规则：*** p<0.01, ** p<0.05, * p<0.1"""
    if not np.isfinite(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.1:
        return "*"
    return ""

def fmt_num(x, digits=4):
    """数值格式化（去尾零）"""
    if not np.isfinite(x):
        return ""
    return f"{x:.{digits}f}".rstrip("0").rstrip(".")

def fmt_ci(lo, hi, digits=4):
    """95%CI格式化"""
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return ""
    return f"{fmt_num(lo,digits)}–{fmt_num(hi,digits)}"

tmp = res_all.copy()

# 星号
tmp["star"] = tmp["p"].map(sig_star)

# 版本A：仅 Coef + star
tmp["coef_star"] = tmp.apply(lambda r: f"{fmt_num(r['Coef'])}{r['star']}", axis=1)

# 版本B：Coef(95%CI) + star
tmp["coef_ci_star"] = tmp.apply(
    lambda r: f"{fmt_num(r['Coef'])}{r['star']} ({fmt_ci(r['CI_lo'], r['CI_hi'])})",
    axis=1
)

In [127]:
# Cell 3：做“13×9矩阵表”（两种版本）
# 你如果已经定义了 X_COLS（9个变量名），会按该顺序排列列
col_order = X_COLS if "X_COLS" in globals() else sorted(tmp["Predictor"].unique())

# ---- 矩阵A：Coef+星号 ----
coef_matrix = (
    tmp.pivot(index="AMR", columns="Predictor", values="coef_star")
       .reindex(columns=col_order)
)

coef_matrix.index.name = None   # 列索引名
# coef_matrix.columns.name = None # 行索引名

# ---- 矩阵B：Coef(95%CI)+星号 ----
# coef_ci_matrix = (
#     tmp.pivot(index="AMR", columns="Predictor", values="coef_ci_star")
#        .reindex(columns=col_order)
# )

print("矩阵A（Coef+星号）预览：")
display(coef_matrix)

# print("矩阵B（Coef(95%CI)+星号）预览：")
# display(coef_ci_matrix)

矩阵A（Coef+星号）预览：


Predictor,省平均气温,省平均降水,R1xday,PM2.5,医疗水平,GDP,城市用水普及率,生活垃圾无害化处理率,抗菌药物使用强度
3GCREC,0.6111,-1.2997,0.6942,1.0761*,-0.6208,0.8532,-1.7732,0.8452**,0.8811**
3GCRKP,3.9611**,-0.7971,1.9578*,2.5087*,1.4493,2.2059*,-1.2195,-0.0612,1.1725**
CRAB,7.1968***,-4.1317,-0.6628,3.8147**,0.6418,2.8114,-3.7126,-0.5713,2.4607***
CREC,0.1532,-0.1292,0.1429*,0.056,0.2207,0.1998*,0.0548,-0.0192,0.1275**
CRKP,1.8312,-0.8289,0.9973,-0.1797,2.1258,1.8257,0.738,0.0934,0.7726
CRPA,1.7308,-0.9649,0.9347,1.0196,2.3305***,1.5729,1.1497,0.2508,0.2091
ERSP,0.7168,-0.54,0.2638,0.5034,0.8759**,1.0225***,-0.0816,0.9587***,-0.1165
MRCNS,0.2456,0.4453,-0.286,0.3748,1.7994***,1.2659**,-1.0782,0.2793,0.5281
MRSA,-0.6461,0.6094,-0.1907,-2.0638,1.0735,3.1468***,1.3742,0.4432,1.0061
PRSP,0.0981,-0.5417,0.2716,-0.221,-0.4354**,-0.2563,0.398,0.2703,0.2457
